# Lecture 19 - Scikit-Learn and Scripting 
 

# Lecture 19 - Scikit-Learn and Scripting 
 
In this lecture we'll cover more advanced topics in Python programming. 
 
 We'll start with Scikit-Learn, a popular machine learning library in Python.

 Then we'll cover scripting in Python, which is a powerful way to write more complex programs.

This tutorial builds on the excellent how-to: https://docs.python.org/3/howto/argparse.html

### Overview


- Introduction to Scikit-Learn
    - Overview of Scikit-Learn
    - Pipeline example

 - Scripting in Python
    - Introduce `__main__`
    - Argument parsing with argparse
 - Example script

## Introduction to Scikit-Learn

Scikit-Learn is a popular machine learning library in Python. It provides simple and efficient tools for data mining and data analysis, built on top of NumPy, SciPy, and Matplotlib.


What is scikit-learn?
- Lightweight, consistent ML library built on NumPy / SciPy / Matplotlib.
- Focus: classical ML (classification, regression, clustering, preprocessing, model selection).
- Design goals: simple, consistent API; composability (pipelines); reproducible experiments.


# Core building blocks
- Estimator: any object with fit(X, y=None) that learns from data.
- Transformer: estimator with transform(X) (optionally fit_transform) — for preprocessing.
- Predictor: estimator with predict(X) (and often predict_proba, decision_function).
- Meta-estimators: wrappers like GridSearchCV, cross_val_score, and pipelines that take other estimators.

We'll see an example of using Scikit-Learn pipelines later in the lecture.

## The Estimator API (conventions)
- Constructor sets hyperparameters only (no data): clf = SVC(C=1.0, kernel='rbf').
- fit(X, y) learns and returns self. Learned attributes end with underscore: coef_, classes_, n_iter_.
- predict(X) / predict_proba(X) for inference; transform(X) for feature ops.
- get_params()/set_params() allow programmatic tuning and cloning.
- Inputs: typically NumPy arrays or array-like (shape (n_samples, n_features)); many estimators accept sparse arrays.


In [ ]:
# simple example using an sklearn estimator
import numpy as np
from sklearn.linear_model import LinearRegression

# create a simple dataset
X = np.array([[1], [2], [3], [4]])
y = np.array([2, 4, 6, 8])

# create and fit the model
model = LinearRegression()
model.fit(X, y)

# make predictions
predictions = model.predict(X)
print(predictions)

## Data Preprocessing in Scikit-Learn

Data preprocessing is a crucial step in the machine learning pipeline. It involves transforming raw data into a format that is suitable for training machine learning models. Scikit-Learn provides a variety of tools to handle common preprocessing tasks, such as:

- **Scaling and Normalization**: Standardize features by removing the mean and scaling to unit variance using `StandardScaler` or normalize feature vectors using `Normalizer`.
- **Encoding Categorical Variables**: Convert categorical data into numerical format using `OneHotEncoder` or `LabelEncoder`.
- **Handling Missing Values**: Fill missing values with a specific value or a statistical measure (mean, median, etc.) using `SimpleImputer`.
- **Feature Transformation**: Apply mathematical transformations to features, such as polynomial features using `PolynomialFeatures`.

### Why is Preprocessing Important?

1. **Improves Model Performance**: Many machine learning algorithms perform better when the data is scaled or normalized.
2. **Handles Data Inconsistencies**: Deals with missing values, outliers, and categorical variables.
3. **Ensures Compatibility**: Some algorithms require specific input formats, such as numerical data.

By using Scikit-Learn's preprocessing tools, you can create a robust and efficient pipeline for preparing your data for machine learning tasks.

### Data preprocessing example

Here is an example of using Scikit-Learn for data preprocessing:


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer

# Sample data
data = pd.DataFrame({
    'num1': [1.0, 2.0, np.nan, 4.0],
    'cat':  ['A', 'B', 'A', 'B'],
    'num2': [10.0, np.nan, 30.0, 40.0]
})

# Define feature groups
numerical_features = ['num1', 'num2']
categorical_features = ['cat']

# Define transformations
numerical_transformer = make_pipeline(
    SimpleImputer(strategy='mean'),
    StandardScaler()
)
categorical_transformer = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder()
)

# Combine them
preprocessor = ColumnTransformer([
    ('num', numerical_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

# Apply
preprocessed = preprocessor.fit_transform(data)
print(preprocessed)

## Composition: Pipelines & ColumnTransformer
- Pipeline: chain transformers then an estimator — use make_pipeline or Pipeline([...]).
    - Ensures correct order, tidy code, and safe hyperparameter search.
- ColumnTransformer: apply different preprocessing to subsets of columns (numerical vs categorical).
- Useful pattern: pipeline = make_pipeline(ColumnTransformer(...), StandardScaler(), estimator)
    - Then pass pipeline to GridSearchCV or cross_val_score.

### Pipeline example
Here is a simple example of using Scikit-Learn to create a machine learning pipeline for classification:


In [ ]:

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

# Load the iris dataset
iris = load_iris()
X, y = iris.data, iris.target
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Note the different averages of each of the features
X_train.mean(axis=0)

In [ ]:

# Create a pipeline with a scaler and an SVM classifier
pipeline = make_pipeline(StandardScaler(), SVC())

# Fit the pipeline to the training data
pipeline.fit(X_train, y_train)

# Predict the labels for the testing data
y_pred = pipeline.predict(X_test)

# Print the classification report
print(classification_report(y_test, y_pred))

## Model selection & practical tips
- Model selection: cross_val_score, GridSearchCV, RandomizedSearchCV; supply cv splitters and scoring.
- Use Pipeline + GridSearchCV to avoid leakage (fit/transform inside CV).
- Common preprocessing: StandardScaler, OneHotEncoder, SimpleImputer.
- Practical tips:
    - Set random_state for reproducibility.
    - Check learned attributes after fit (e.g., classes_, feature_importances_).
    - Use joblib.dump/load for model persistence.
    - Read docs for input dtype/shape requirements and memory use for large / sparse data.
- Resources: official docs (scikit-learn.org) — excellent API reference and examples.

## Model Selection and Evaluation

Scikit-Learn provides various tools for model selection and evaluation, such as cross-validation, grid search, and performance metrics. These tools help in selecting the best model and tuning hyperparameters for optimal performance.

The simplest example of cross-validation is as follows:


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris
from sklearn.svm import SVC

# Load the iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# Create an SVM classifier
svm = SVC()

# Perform 5-fold cross-validation
scores = cross_val_score(svm, X, y, cv=5)

# Print the cross-validation scores
print("Cross-validation scores:", scores)
print("Mean cross-validation score:", scores.mean())

## Scripting

Notebooks are amazing for interactive scripting, data exploration and simple analysis. 

There are many occasions however when it's more useful to be able to script your analysis. For example:
 - Long running processes / analyses, such as training a deep learning model
 - Running multiple variations on an analysis
 - Others??
 
 Class[Buzz]: `script_examples`

### Execution

Let's take a look at a very simple script: [my_module.py](my_module.py).

We can run a python module like this from the command line easily:

In [ ]:
! python my_module.py

On linux and macOS machines we can specify that the file should be run with Python and execute it directly

We just need to add a 'shebang' at the top of our script: `!/usr/bin/env python3`

Now we can run it directly:

In [ ]:
! ./my_module_x.py

One problem with this approach is that if we import the module it will run that code every time. This probably isn't what we want...

In [ ]:
import my_module_x

Python uses a special name for a module when it is being executed directly and we can use this to check if the module is being executed or imported:

In [ ]:
if __name__ == '__main__':
    print("I'm in a script")

Let's take a look at [my_module_main.py](my_module_main.py). Now if we import the module we don't get the message printed:

In [ ]:
import my_module_main

In [ ]:
! ./my_module_main.py

### Command line arguments

This is fine for very simple scripts that always perfrom the same actions, but we usually want to be able to alter the behaviour each time we run the script. 

Let's take a look at an example command line program to get an idea of what that looks like: `ls`.

Python provides a very convenient method for managing and parsing command line arguments using the `argparse` library: [my_module_main_args.py](my_module_main_args.py)

In [ ]:
! ./my_module_main_args.py -l

This simple setup already provides some useful funcionality:
 - Includes a help description describing correct usage
 - Catches incorrect arguments

How do we add more useful arguments? And how do we access those?

There are a few options, but the builtin `argparse` library provides the most flexibility pretty easily:

In [ ]:
import argparse

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("echo")  # Declare a (required) argument called echo
args = parser.parse_args(['hello'])  # Pretend we just called the program like: my_module_main_args.py foo
print(args)

In [ ]:
args.echo

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("echo", help="echo the string you use here")
parser.print_help() # my_script.py --help

OK, let's try something a bit more useful:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("square", help="display a square of a given number")
args = parser.parse_args(['2'])
print(args.square**2)

Let's fix that:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("square", help="display a square of a given number",
                    type=int)
args = parser.parse_args(['2'])
print(args.square**2)

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("square", help="display a square of a given number",
                    type=int)
args = parser.parse_args(['four'])
print(args.square**2)

We can also include optional arguments using the '--' prefix:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--verbosity", help="increase output verbosity")
args = parser.parse_args(['--verbosity=true'])
if args.verbosity:
    print("verbosity turned on")


In [ ]:
args = parser.parse_args(['--verbosity', '1'])
if args.verbosity:
    print("verbosity turned on")


Sometimes we might not want any particular value, but just a flag. We can do that using the `action` keyword:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--verbose", help="increase output verbosity",
                    action="store_true")
args = parser.parse_args([])
if args.verbose:
    print("verbosity turned on")

We can even provide a shorthand:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("-v", "--verbose", help="increase output verbosity",
                    action="store_true")
args = parser.parse_args(['-v'])
if args.verbose:
    print("verbosity turned on")

Combining these arguments is easy:

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("square", type=int,
                    help="display a square of a given number")
parser.add_argument("-v", "--verbose", action="store_true",
                    help="increase output verbosity")

In [ ]:
args = parser.parse_args(['2'])
answer = args.square**2
if args.verbose:
    print(f"the square of {args.square} equals {answer}")
else:
    print(answer)


In [ ]:
args = parser.parse_args(['2', '-v'])
answer = args.square**2
if args.verbose:
    print(f"the square of {args.square} equals {answer}")
else:
    print(answer)


Here's an example of a (simple) complete script: [example_script.py](example_script.py)

We've include multiple arguments of a particular type and a 'countable' optional argument.

## Exercise 1.

Work in pairs to create a script that accepts multiple commands to:
 - Print the contents of a NetCDF file
 - Calculate the time average, and writes the result to an output file
 - Provide a useful error message if the file doesn't exist, or xarray can't open it

There is an example NetCDF file in the public DataHub directory.

Solution: [nc.py](nc.py)